# TabularTransformer — Supervised Training

The canonical training notebook for `credit-default-tabular-transformer`. Runs identically under:

* **Google Colab** — cloud GPU (T4 / L4 / A100). Auto-detects the runtime, clones the repo, installs minimal deps.
* **VS Code Colab extension** — kernel runs on Colab's backend; same auto-detection path (`"google.colab" in sys.modules`).
* **Local Jupyter / VS Code** — uses the repo you already have, defers dependency management to your existing poetry env.

**Depth of integration.** Every cell imports directly from `src/` — zero re-implementation.

| Cell | Project module used |
| --- | --- |
| 1 | Runtime detection + lazy deps |
| 2 | `attention`, `dataset`, `embedding`, `losses`, `model`, `tokenizer`, `transformer`, `utils` — every src/ module + drift guards for TOKEN_ORDER and both novelty-bias layouts |
| 3 | `run_pipeline.py --preprocess-only` shell-out |
| 4 | `tokenizer.CreditDefaultDataset`, `dataset.make_loader` |
| 5 | `model.TabularTransformer` — direct instantiation with every plan switch |
| 6 | `model.TabularTransformer.summary()` — architecture + parameter breakdown |
| 7 | `matplotlib` bar chart of `parameter_count_by_module()` |
| 8 | `train.main([...])` — full CLI semantics from Python |
| 9 | Training-curve panel from `train_log.csv` |
| 10 | `utils.load_checkpoint` (weights-only) + `train.evaluate_on_loader` |
| 11 | Reliability diagram (calibration preview) |
| 12 | Multi-seed sweep driver (Plan §14.1) |
| 13 | Ablation grid driver (A5 × A22) |
| 14 | [CLS]→feature attention bar chart — colour-coded by feature group |

**Plan alignment**: §6 architecture · §7 focal loss · §8 AdamW + cosine warmup + early stop · §8.5.5 fine-tune from MTLM · **Novelty N2** feature-group attention bias (A21) · **Novelty N3** temporal-decay bias (A22) · **Novelty N5** multi-task PAY_0 aux (A16).

## 1. Environment bootstrap

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path


def _detect_runtime() -> str:
    """Return one of 'colab', 'vscode-local', 'jupyter-local'."""
    if "google.colab" in sys.modules:
        return "colab"
    if os.environ.get("VSCODE_PID") or os.environ.get("TERM_PROGRAM") == "vscode":
        return "vscode-local"
    return "jupyter-local"


RUNTIME = _detect_runtime()
print(f"Runtime: {RUNTIME}")

REPO_URL = "https://github.com/abailey81/credit-default-tabular-transformer.git"

if RUNTIME == "colab":
    repo_dir = Path("/content/credit-default-tabular-transformer")
    if not repo_dir.exists():
        print(f"Cloning {REPO_URL} …")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)
else:
    repo_dir = Path.cwd()
    while not (repo_dir / "pyproject.toml").is_file():
        if repo_dir.parent == repo_dir:
            raise RuntimeError(
                "Could not find repo root (no pyproject.toml found up the tree). "
                "Start the notebook from inside the credit-default-tabular-transformer clone."
            )
        repo_dir = repo_dir.parent
    os.chdir(repo_dir)

REQUIRED_PKGS = {
    "torch":      "torch",       "numpy":       "numpy",
    "pandas":     "pandas",      "sklearn":     "scikit-learn",
    "matplotlib": "matplotlib",  "seaborn":     "seaborn",
    "xlrd":       "xlrd",        "openpyxl":    "openpyxl",
    "ucimlrepo":  "ucimlrepo",
}
missing = []
for mod, pkg in REQUIRED_PKGS.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pkg)
if missing:
    if RUNTIME == "colab":
        print(f"Installing missing deps on Colab: {missing}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    else:
        raise RuntimeError(
            f"Missing Python packages: {missing}. Run `poetry install` from the repo root."
        )

src_dir = str(repo_dir / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import torch
print(f"repo root:       {repo_dir}")
print(f"torch:           {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")

## 2. Project sanity check — full-module imports + drift guards

Imports every `src/` module and confirms the canonical `TOKEN_ORDER`, `TemporalDecayBias` layout (N3), and `FeatureGroupBias` assignment (N2) haven't silently drifted. Also prints a live table of what feature sits in each token position — useful as a quick reference throughout the rest of the notebook.

In [ ]:
from attention import MultiHeadAttention, ScaledDotProductAttention
from dataset import StratifiedBatchSampler, make_loader
from embedding import (
    CAT_VOCAB_SIZES,
    FEATURE_GROUP_NAMES,
    FeatureEmbedding,
    N_FEATURE_GROUPS,
    TOKEN_ORDER,
    build_group_assignment,
    build_temporal_layout,
    describe_token_layout,
)
from losses import FocalLoss, LabelSmoothingBCELoss, WeightedBCELoss, balanced_alpha
from model import TabularTransformer
from tokenizer import (
    CreditDefaultDataset,
    MTLMCollator,
    PAY_RAW_NUM_CLASSES,
    PAY_STATE_NAMES,
    build_categorical_vocab,
)
from transformer import (
    FeatureGroupBias,
    FeedForward,
    TemporalDecayBias,
    TransformerBlock,
    TransformerEncoder,
)
from utils import (
    EarlyStopping,
    configure_logging,
    count_parameters,
    format_parameter_count,
    get_device,
    load_checkpoint,
    save_checkpoint,
    set_deterministic,
)

configure_logging()

# Drift guards — catch silent TOKEN_ORDER / group reordering before it causes harm.
assert len(TOKEN_ORDER) == 23
assert PAY_RAW_NUM_CLASSES == 11
assert N_FEATURE_GROUPS == 5
expected_temporal = {
    "pay":     {"positions": [4, 5, 6, 7, 8, 9],         "months": list(range(6))},
    "bill":    {"positions": [12, 13, 14, 15, 16, 17],   "months": list(range(6))},
    "pay_amt": {"positions": [18, 19, 20, 21, 22, 23],   "months": list(range(6))},
}
assert build_temporal_layout() == expected_temporal, "TOKEN_ORDER temporal drift"
expected_groups = [0, 1, 1, 1, 2, 2, 2, 2, 2, 2, 1, 1, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4]
assert build_group_assignment() == expected_groups, "Feature-group drift"

print("All project modules imported; drift guards passed.")
print(f"Categorical vocabularies: {CAT_VOCAB_SIZES}")
print(f"PAY state names:          {PAY_STATE_NAMES}")
print(f"Feature groups:           {FEATURE_GROUP_NAMES}")
print()
print(describe_token_layout())

## 3. Preprocessing (idempotent)

In [ ]:
required_csvs = [
    "data/processed/feature_metadata.json",
    "data/processed/train_scaled.csv",
    "data/processed/val_scaled.csv",
    "data/processed/test_scaled.csv",
]
missing_csvs = [p for p in required_csvs if not Path(p).is_file()]
if missing_csvs:
    print(f"Regenerating preprocessing outputs (missing: {missing_csvs}) …")
    result = subprocess.run(
        [sys.executable, "run_pipeline.py", "--preprocess-only", "--source", "local"],
        cwd=str(repo_dir),
        env={**os.environ, "PYTHONIOENCODING": "utf-8"},
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        raise RuntimeError("Preprocessing failed.")
    print("Preprocessing complete.")
else:
    print("All preprocessing outputs present; skipping.")

## 4. Dataset + DataLoaders

In [ ]:
import json

import pandas as pd

meta = json.loads(Path("data/processed/feature_metadata.json").read_text())
cat_vocab = build_categorical_vocab(meta)

train_df = pd.read_csv("data/processed/train_scaled.csv")
val_df   = pd.read_csv("data/processed/val_scaled.csv")
test_df  = pd.read_csv("data/processed/test_scaled.csv")

train_ds = CreditDefaultDataset(train_df, cat_vocab, verbose=False)
val_ds   = CreditDefaultDataset(val_df,   cat_vocab, verbose=False)
test_ds  = CreditDefaultDataset(test_df,  cat_vocab, verbose=False)

for split_name, ds in (("train", train_ds), ("val", val_ds), ("test", test_ds)):
    pos_rate = float(ds.tensors["labels"].mean())
    print(f"{split_name:5s}: {len(ds):>6,d} rows | default rate = {pos_rate:.4f}")

## 5. Instantiate TabularTransformer

Builds the model directly (not via `train.main`) so you can inspect its structure before training. Every Plan §6 / §11 switch is available as a kwarg.

In [ ]:
set_deterministic(seed=42, warn_only=True)
device = get_device("auto")

MODEL_KWARGS = dict(
    d_model=32,
    n_heads=4,
    n_layers=2,
    dropout=0.1,
    classification_dropout=0.1,
    pool="cls",                           # Ablation A5: 'cls' | 'mean' | 'max'
    use_temporal_pos=True,                # Ablation A7
    temporal_decay_mode="scalar",         # Novelty N3 / Ablation A22
    feature_group_bias_mode="scalar",     # Novelty N2 / Ablation A21
    aux_pay0=False,                       # Novelty N5 / Ablation A16
    cat_vocab_sizes={
        feat: info["n_categories"] for feat, info in meta["categorical_features"].items()
    },
)

preview_model = TabularTransformer(**MODEL_KWARGS).to(device)

preview_loader = make_loader(train_ds, batch_size=32, mode="val")
batch = next(iter(preview_loader))
batch = {k: (v.to(device) if torch.is_tensor(v) else
             {kk: vv.to(device) for kk, vv in v.items()} if isinstance(v, dict) else v)
         for k, v in batch.items()}
preview_model.eval()
with torch.no_grad():
    out = preview_model(batch, return_attn=True)
print(f"Device:                 {device}")
print(f"Forward logit shape:    {tuple(out['logit'].shape)}")
print(f"Attention layers:       {len(out['attn_weights'])}")
print(f"Per-layer attn shape:   {tuple(out['attn_weights'][0].shape)}")
print(f"repr: {preview_model!r}")

## 6. `model.summary()` — sophisticated architecture print

Renders a per-module parameter breakdown plus every active ablation-switch state — pulled live from the model object so a misconfiguration shows up immediately. Useful as a one-shot audit before committing to a training run.

In [ ]:
print(preview_model.summary())

## 7. Parameter-breakdown visualisation

In [ ]:
import matplotlib.pyplot as plt

breakdown = preview_model.parameter_count_by_module()
labels, counts = zip(*sorted(breakdown.items(), key=lambda kv: -kv[1]))
total = sum(counts)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(labels, counts, color="#534AB7")
ax.set_xlabel("Trainable parameters")
ax.set_title(f"TabularTransformer parameter breakdown — {format_parameter_count(total)} total")
for bar, count in zip(bars, counts):
    pct = 100.0 * count / total
    ax.text(bar.get_width() + total * 0.005, bar.get_y() + bar.get_height()/2,
            f"  {count:,d} ({pct:.1f}%)", va="center", fontsize=9)
ax.invert_yaxis()
plt.tight_layout(); plt.show()

del preview_model, preview_loader, out
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 8. Run training (full plan-aligned pipeline)

Defers to `train.main` — AdamW (§8.1), cosine-warmup LR (§8.2), grad clipping (§8.3), stratified batching (§8.8), early stopping on val AUC-ROC (§8.5), optional two-stage LR from MTLM (§8.5.5), optional multi-task PAY_0 (§8.6 / N5), checkpoint + per-epoch CSV + test metrics + threshold sweep + (optional) attention dump.

In [ ]:
from train import main as train_main

SEED = 42
OUTPUT_DIR = f"results/transformer/seed_{SEED}"

ARGV = [
    "--seed", str(SEED),
    "--output-dir", OUTPUT_DIR,
    "--device", "auto",
    "--determinism",

    "--d-model", str(MODEL_KWARGS["d_model"]),
    "--n-heads", str(MODEL_KWARGS["n_heads"]),
    "--n-layers", str(MODEL_KWARGS["n_layers"]),
    "--pool", MODEL_KWARGS["pool"],
    "--temporal-decay-mode", MODEL_KWARGS["temporal_decay_mode"],
    "--feature-group-bias-mode", MODEL_KWARGS["feature_group_bias_mode"],
    *(["--use-temporal-pos"] if MODEL_KWARGS["use_temporal_pos"] else []),

    "--dropout", str(MODEL_KWARGS["dropout"]),
    "--classification-dropout", str(MODEL_KWARGS["classification_dropout"]),

    "--loss", "focal",
    "--focal-gamma", "2.0",
    "--focal-alpha", "balanced",

    "--lr", "3e-4",
    "--weight-decay", "1e-5",
    "--warmup-frac", "0.10",
    "--min-lr-frac", "0.01",
    "--grad-clip", "1.0",

    "--epochs", "200",
    "--patience", "20",
    "--batch-size", "256",
    "--stratified-batches",
]

rc = train_main(ARGV)
assert rc == 0
print(f"\nTraining complete → {OUTPUT_DIR}")

## 9. Training curves

In [ ]:
log = pd.read_csv(f"{OUTPUT_DIR}/train_log.csv")

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(log["epoch"], log["train_loss"], color="#534AB7")
axes[0, 0].set_title("Training loss (focal)"); axes[0, 0].set_xlabel("epoch"); axes[0, 0].set_ylabel("loss")
axes[0, 0].grid(alpha=0.3)
axes[0, 1].plot(log["epoch"], log["val_auc_roc"], color="#1D9E75", label="val AUC-ROC")
axes[0, 1].plot(log["epoch"], log["val_auc_pr"],  color="#D85A30", label="val AUC-PR")
axes[0, 1].set_title("Validation discrimination"); axes[0, 1].set_xlabel("epoch")
axes[0, 1].set_ylim(0.5, 1.0); axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)
axes[1, 0].plot(log["epoch"], log["lr"], color="#6B7280")
axes[1, 0].set_title("LR schedule (cosine + warmup)"); axes[1, 0].set_xlabel("epoch"); axes[1, 0].set_ylabel("LR")
axes[1, 0].grid(alpha=0.3)
axes[1, 1].plot(log["epoch"], log["grad_norm_mean"], color="#378ADD", label="mean")
axes[1, 1].plot(log["epoch"], log["grad_norm_max"],  color="#D85A30", label="max")
axes[1, 1].set_title("Gradient-norm tracking"); axes[1, 1].set_xlabel("epoch")
axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 10. Re-evaluate checkpoint from disk

Round-trip through `utils.load_checkpoint` (weights-only / SECURITY_AUDIT-hardened) + `train.evaluate_on_loader`. Proves the checkpoint is self-contained and the inference path is reproducible.

In [ ]:
from train import compute_classification_metrics, evaluate_on_loader

fresh_model = TabularTransformer(**MODEL_KWARGS).to(device)
load_checkpoint(
    f"{OUTPUT_DIR}/best.pt", fresh_model,
    strict=True, trust_source=False, map_location=device,
)

test_loader = make_loader(test_ds, batch_size=256, mode="test")
rerun = evaluate_on_loader(fresh_model, test_loader, device=device)

print("Test metrics (re-evaluated from disk):")
for k, v in rerun["metrics"].items():
    print(f"  {k:10s}  {v:.4f}")

saved = json.loads(Path(f"{OUTPUT_DIR}/test_metrics.json").read_text())["metrics"]
max_delta = max(abs(rerun["metrics"][k] - saved[k]) for k in saved if saved[k] == saved[k])
print(f"\nMax |re-eval − saved| across metrics: {max_delta:.2e}")
assert max_delta < 1e-5, "checkpoint round-trip changed the test metrics"

## 11. Reliability diagram (calibration preview)

15-bin reliability curve showing the gap between predicted probability and observed default rate — plus a confidence histogram. Full calibration analysis (temperature scaling, Brier decomposition, subgroup ECE) ships with `src/calibration.py` in a later PR. Plan §13.

In [ ]:
import numpy as np

preds = np.load(f"{OUTPUT_DIR}/test_predictions.npz")
y_true = preds["y_true"]; y_prob = preds["y_prob"]

N_BINS = 15
edges = np.linspace(0.0, 1.0, N_BINS + 1)
centres, accs, confs, weights = [], [], [], []
for i in range(N_BINS):
    lo, hi = edges[i], edges[i + 1]
    mask = (y_prob >= lo) & ((y_prob < hi) if i < N_BINS - 1 else (y_prob <= hi))
    if mask.sum() == 0:
        continue
    centres.append((lo + hi) / 2.0)
    accs.append(float(y_true[mask].mean()))
    confs.append(float(y_prob[mask].mean()))
    weights.append(int(mask.sum()))
ece = sum(w / len(y_prob) * abs(a - c) for w, a, c in zip(weights, accs, confs))

fig, (ax, axw) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [2, 1]})
ax.plot([0, 1], [0, 1], ls="--", color="#6B7280", label="Perfect calibration")
ax.plot(confs, accs, "o-", color="#534AB7", markersize=7, linewidth=2, label="TabularTransformer")
ax.set_xlabel("Predicted probability (bin mean)")
ax.set_ylabel("Observed default rate")
ax.set_title(f"Reliability diagram — ECE = {ece:.4f} on test set")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(alpha=0.3); ax.legend()
axw.bar(centres, weights, width=(1.0 / N_BINS) * 0.9, color="#378ADD", alpha=0.7)
axw.set_xlim(0, 1); axw.set_xlabel("Predicted probability"); axw.set_ylabel("Test rows per bin")
axw.set_title("Prediction-confidence histogram"); axw.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 12. Multi-seed sweep (Plan §14.1)

Plan §14.1 requires **5 seeds** for the final report. Each seed writes its own `results/transformer/seed_{s}/`. `src/evaluate.py` (later PR) aggregates with paired-bootstrap CIs (Plan §14.5). Uncomment the final lines to run — ~15-25 minutes on a T4 for 5 seeds.

In [ ]:
def run_seed_sweep(seeds, extra_argv=None, verbose=True):
    """Train once per seed and aggregate the test metrics into a DataFrame."""
    extra_argv = extra_argv or []
    # Strip --seed / --output-dir from the base ARGV so we can override.
    base, skip_next = [], False
    for tok in ARGV:
        if skip_next:
            skip_next = False
            continue
        if tok in ("--seed", "--output-dir"):
            skip_next = True
            continue
        base.append(tok)

    rows = []
    for s in seeds:
        out_dir = f"results/transformer/seed_{s}"
        argv = [*base, "--seed", str(s), "--output-dir", out_dir, *extra_argv]
        if verbose:
            print(f"\n=== seed {s} → {out_dir} ===")
        train_main(argv)
        tm = json.loads(Path(f"{out_dir}/test_metrics.json").read_text())
        rows.append({"seed": s, **tm["metrics"]})
    return pd.DataFrame(rows)

# Uncomment to actually run the 5-seed sweep:
# seed_df = run_seed_sweep([1, 2, 3, 4, 5])
# print(seed_df.to_string(index=False))
# print("\nMean ± std across seeds:")
# for col in ("auc_roc", "auc_pr", "f1", "ece", "brier"):
#     print(f"  {col:10s}  {seed_df[col].mean():.4f} ± {seed_df[col].std():.4f}")

## 13. Ablation grid (A5 × A22)

Minimal grid demonstrating how to drive Plan §11 ablations from the notebook. Each row calls `train.main` with a different config and summarises the test-set AUC-ROC. The full 22-ablation matrix (A1–A22 + BH-FDR correction) is driven by `src/ablation_runner.py` in a later PR — this cell is for quick on-notebook exploration.

In [ ]:
def run_ablation_grid(grid, seed=42, epochs=50, patience=10, verbose=True):
    """Run a list of configs via train.main and collect test metrics."""
    rows = []
    for idx, cfg in enumerate(grid):
        out_dir = f"results/transformer/ablation_{idx:02d}"
        argv = [
            "--seed", str(seed),
            "--output-dir", out_dir,
            "--device", "auto",
            "--d-model", "32", "--n-heads", "4", "--n-layers", "2",
            "--lr", "3e-4", "--weight-decay", "1e-5",
            "--batch-size", "256",
            "--epochs", str(epochs), "--patience", str(patience),
            "--loss", "focal", "--focal-gamma", "2.0", "--focal-alpha", "balanced",
            "--no-save-attn",
        ]
        for k, v in cfg.items():
            flag = "--" + k.replace("_", "-")
            if isinstance(v, bool):
                if v:
                    argv.append(flag)
            else:
                argv.extend([flag, str(v)])
        if verbose:
            print(f"\n=== ablation {idx}: {cfg} ===")
        train_main(argv)
        tm = json.loads(Path(f"{out_dir}/test_metrics.json").read_text())
        rows.append({**cfg, **{f"test_{k}": v for k, v in tm["metrics"].items()}})
    return pd.DataFrame(rows)

# Example grid: pool × temporal-decay. Uncomment to run on Colab.
# grid = [
#     {"pool": "cls",  "temporal_decay_mode": "off"},
#     {"pool": "cls",  "temporal_decay_mode": "scalar"},
#     {"pool": "cls",  "temporal_decay_mode": "per_head"},
#     {"pool": "mean", "temporal_decay_mode": "off"},
#     {"pool": "mean", "temporal_decay_mode": "scalar"},
#     {"pool": "max",  "temporal_decay_mode": "off"},
# ]
# ablation_df = run_ablation_grid(grid, epochs=30, patience=10)
# print(ablation_df.sort_values("test_auc_roc", ascending=False).to_string(index=False))

## 14. [CLS]→feature attention (colour-coded by group)

Quick bar chart of the final-layer [CLS]→feature attention averaged over the test set, colour-coded by `FEATURE_GROUP_NAMES`. Full Plan §12 rollout / Jain-&-Wallace diagnostic (Novelty N8) / per-head entropy / attention-by-class ship with `src/interpret.py` later.

In [ ]:
attn_path = Path(f"{OUTPUT_DIR}/test_attn_weights.npz")
if attn_path.is_file():
    attn = np.load(attn_path)
    last_key = sorted(attn.keys())[-1]
    last_layer = attn[last_key]                                       # (N_test, n_heads, 24, 24)
    cls_to_features = last_layer[:, :, 0, 1:].mean(axis=(0, 1))       # (23,)

    group_assignment_23 = build_group_assignment(cls_offset=0)
    palette = {1: "#1D9E75", 2: "#D85A30", 3: "#378ADD", 4: "#534AB7"}
    colors = [palette[g] for g in group_assignment_23]

    fig, ax = plt.subplots(figsize=(13, 4))
    ax.bar(range(len(TOKEN_ORDER)), cls_to_features, color=colors)
    ax.set_xticks(range(len(TOKEN_ORDER)))
    ax.set_xticklabels(TOKEN_ORDER, rotation=45, ha="right")
    ax.set_ylabel("Mean attention from [CLS]")
    ax.set_title(f"Final-layer ({last_key}) [CLS]→feature attention — colour-coded by group")
    from matplotlib.patches import Patch
    legend_entries = [
        Patch(color=palette[1], label=FEATURE_GROUP_NAMES[1]),
        Patch(color=palette[2], label=FEATURE_GROUP_NAMES[2]),
        Patch(color=palette[3], label=FEATURE_GROUP_NAMES[3]),
        Patch(color=palette[4], label=FEATURE_GROUP_NAMES[4]),
    ]
    ax.legend(handles=legend_entries, loc="upper right")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("test_attn_weights.npz not present — pass --save-attn (default) during training.")